# Build and plot a full-sky noise map

This notebook downgrades, stitches, saves, and plots one selected Q/U noise realization from the `v4_10_arcmin` patch files.

## Imports and choices

Change the seed values or `USE_WEIGHTED` here before running the notebook.

In [ ]:
import sys
from pathlib import Path

import os
import re
from collections import defaultdict

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from astropy.io import fits

TARGET_NSIDE = 1024
BORDERPIX = 64
KEEPBORDER = 32
SUBDIVIDE = 4
USE_WEIGHTED = False
NOISE_SEED = 296
CMB_RES_SEED = 96
FREQUENCY = 353
NUISANCE_VERSION = "v4_10_arcmin"

NUISANCE_DIR = Path("/pscratch/sd/e/erussie/GNILC+ST/patches/nuisance")
RESOURCES_DIR = Path("/path/to/resources/")
OUTPUT_DIR = Path("/path/to/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PIXEL_FACES = RESOURCES_DIR / f"pixfaces_nside={TARGET_NSIDE}_subdivide={SUBDIVIDE}_borderpix={BORDERPIX}.fits"
NOISE_FITS = OUTPUT_DIR / (
    f"noise_Q{FREQUENCY}_U{FREQUENCY}_noise_seed_{NOISE_SEED:04d}_"
    f"CMB_res_seed_{CMB_RES_SEED:02d}_{NUISANCE_VERSION}_nside{TARGET_NSIDE}_weighted={USE_WEIGHTED}.fits"
)

print("Output:", NOISE_FITS)

## Load the patch geometry

The `pixfaces` file maps every pixel in every square patch to a full-sky HEALPix pixel. The same border trimming and optional cosine weighting are used as in `fits_notebook.ipynb`.

In [ ]:
pixfaces_all = fits.getdata(PIXEL_FACES).astype(np.int64)
patch_indices = np.arange(pixfaces_all.shape[0])
full_height, full_width = pixfaces_all.shape[1:]
trim = BORDERPIX - KEEPBORDER

pixfaces = pixfaces_all
if trim > 0:
    pixfaces = pixfaces[:, trim:-trim, trim:-trim]
height, width = pixfaces.shape[1:]

weights = np.ones((height, width), dtype=float)
if KEEPBORDER > 0:
    edge = 0.5 * (1.0 - np.cos(np.pi * (np.arange(KEEPBORDER) + 0.5) / KEEPBORDER))
    weights[:KEEPBORDER, :] *= edge[:, None]
    weights[-KEEPBORDER:, :] *= edge[::-1, None]
    weights[:, :KEEPBORDER] *= edge[None, :]
    weights[:, -KEEPBORDER:] *= edge[None, ::-1]

print(f"{len(patch_indices)} patches, stitched patch shape {height} x {width}")

## Load, downgrade, stitch, and save the noise realization

Only files ending in `v4_10_arcmin.npy` are considered. The same `(NOISE_SEED, CMB_RES_SEED)` pair is used for every patch and for both Q and U.

In [ ]:
def load_noise_patch(patch, stokes):
    pattern = (
        f"patch_{patch}_noise_{stokes}{FREQUENCY}_*_noise_seed_{NOISE_SEED:04d}_"
        f"CMB_res_seed_{CMB_RES_SEED:02d}_{NUISANCE_VERSION}.npy"
    )
    files = sorted(NUISANCE_DIR.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No file matching {pattern} in {NUISANCE_DIR}")

    image = np.load(files[0]).astype(float)
    if image.shape == (2 * full_height, 2 * full_width):
        image = image.reshape(full_height, 2, full_width, 2).mean(axis=(1, 3))
    if image.shape == (full_height, full_width) and trim > 0:
        image = image[trim:-trim, trim:-trim]
    if image.shape != (height, width):
        raise ValueError(f"Unexpected {stokes} patch shape after downgrade/trim: {image.shape}")
    return image

q_patches = np.asarray([load_noise_patch(patch, "Q") for patch in patch_indices])
u_patches = np.asarray([load_noise_patch(patch, "U") for patch in patch_indices])

npix = hp.nside2npix(TARGET_NSIDE)
numerator_q = np.zeros(npix)
numerator_u = np.zeros(npix)
denominator = np.zeros(npix)

pixel_index = pixfaces.reshape(-1)
q = q_patches.reshape(-1)
u = u_patches.reshape(-1)
w = np.tile(weights, (len(patch_indices), 1, 1)).reshape(-1) if USE_WEIGHTED else np.ones_like(q)
good = (pixel_index >= 0) & (pixel_index < npix) & np.isfinite(q) & np.isfinite(u) & np.isfinite(w) & (w > 0)

np.add.at(numerator_q, pixel_index[good], q[good] * w[good])
np.add.at(numerator_u, pixel_index[good], u[good] * w[good])
np.add.at(denominator, pixel_index[good], w[good])

covered = denominator > 0
noise_map = np.full((3, npix), hp.UNSEEN, dtype=float)
noise_map[0, covered] = 0.0
noise_map[1, covered] = numerator_q[covered] / denominator[covered]
noise_map[2, covered] = numerator_u[covered] / denominator[covered]

hp.write_map(
    NOISE_FITS,
    noise_map,
    overwrite=True,
    column_names=["I", "Q", "U"],
    column_units=["MJy/sr"] * 3,
)
print(f"Covered pixels: {covered.sum():,}/{npix:,}")
print("Saved:", NOISE_FITS)

## Plot the full-sky noise map

Q and U use a shared symmetric colour scale based on the 99th percentile of their absolute values.

In [ ]:
values = noise_map[1:][np.isfinite(noise_map[1:]) & (noise_map[1:] != hp.UNSEEN)]
limit = 0.02

fig = plt.figure(figsize=(12, 8))
hp.mollview(noise_map[1], fig=fig.number, sub=(2, 1, 1), title=f"Noise Q ({NOISE_SEED:04d}/{CMB_RES_SEED:02d})", unit="MJy/sr", cmap="RdBu_r", min=-limit, max=limit)
hp.mollview(noise_map[2], fig=fig.number, sub=(2, 1, 2), title=f"Noise U ({NOISE_SEED:04d}/{CMB_RES_SEED:02d})", unit="MJy/sr", cmap="RdBu_r", min=-limit, max=limit)
plt.show()